I wrote this notebook with the aim of learning how to compute the reduced density matrix (rdm) and the 
Von Neuman entropy from an MPS.

Considering a simple two-site chain with two spin-1/2 and a Bell state 
$$
    \ket{\Psi} = \frac{\ket{\downarrow_A \downarrow_B}+\ket{\uparrow_A\uparrow_B}}{\sqrt{2}} \Rightarrow \hat{\rho} = \frac{1}{2} \begin{pmatrix} 1 & 0 \\ 0 & 1\end{pmatrix},
$$
the eigenvalues of this matrix are trivially $\lambda = 1/2, 1/2$. 

The  Von Neumann entropy reads $S(\hat{\rho}) = - \text{tr} \left(\hat{\rho} \log \hat{\rho}\right) = - \sum_{k} \lambda^{k} \log \lambda^{k}$. So for the 
state $\ket{\Psi}$ we have that $S(\hat{\rho}) = 1$. 

This shall also be the case for the reduced state of $\hat{\rho}$. For example, considering the partitions $A$ and $B$ we 
have that
$$ 
    \hat{\rho}_A = \text{tr}_{B} (\hat{\rho}) = \sum_{i} \braket{i_{B}| \Psi }\braket{\Psi | i_{B}} = \frac{\ket{\uparrow_A} + \ket{\downarrow_{A}}}{\sqrt{2}}
$$

so the entanglement entropy is simply $S(\hat{\rho}_A) = 1$.

Trying to obtain this result with Itensor and MPS:

In [2]:
using ITensors
using ITensorMPS

using Random

#=
    Generates the MPO for the EHM Hamiltonian 
    with strengths J, U and V. 
    Requires a SiteType sites.
=#
function H_EHM(N, J, U, V, sites)
    os = OpSum()
    for i in 1:(N - 1)
      # Knetic 
      os -= J, "Cdagup", i, "Cup", i + 1
      os -= J, "Cdagup", i + 1, "Cup", i
      os -= J, "Cdagdn", i, "Cdn", i + 1
      os -= J, "Cdagdn", i + 1, "Cdn", i
      # Nearest-neighbours
      os += V, "Ntot", i, "Ntot", i + 1
    end
    # on-site
    for i in 1:N
      os += U, "Nupdn", i
    end
    return MPO(os, sites)
end
function random_metallic_state(L, Nup, Ndn)
    state = fill("Emp", L)
    # Random order of site indices
    up_sites = shuffle(1:L)[1:Nup]
    for i in up_sites
        state[i] = "Up"
    end
    dn_sites = shuffle(1:L)
    added_dn = 0
    for i in dn_sites
        if added_dn == Ndn
            break
        end
        if state[i] == "Emp"
            state[i] = "Dn"
        elseif state[i] == "Up"
            state[i] = "UpDn"
        end
        added_dn += 1
    end
    return state
end

random_metallic_state (generic function with 1 method)

In [1]:
function average_single_site_entanglement_entropy(psi, L)
    # Average entropy taken over all sites. 
    S_avg = 0.0
    # Over all sites j
    for j = 1:L 

        # change orthogonality center to j
        psi = orthogonalize(psi, j)

        # tensor at site j
        A = psi[j]

        # prime the physical index 
        A_dag = dag(A)
        prime!(A_dag, "Site")
        # Diagonalize
        rdm = A * A_dag
        D, U = eigen(rdm)

        # Compute von Neumann entropy safely
        S = 0.0
        for n=1:dim(D, 1)
            p = D[n,n]  # Might be ComplexF64 due to numerical noise
            p_real = real(p)  # Take real part (imaginary part should be ~0)
            if p_real > 1e-15
                S -= p_real * log(p_real)
            end
        end
        S_avg += S 
    end
    return S_avg / L 
end

average_single_site_entanglement_entropy (generic function with 1 method)

In [3]:
L = 10

sites = siteinds("Electron", L; conserve_qns=true)

maxdim = [50, 100, 200, 400, 800, 800]
cutoff = [1E-14]

nsweeps = 6

Npart = floor(Int, L/2) 
Nup = Npart + L % 2 
Ndn = L - Nup 

state = random_metallic_state(L, Nup, Ndn)

psi0 = random_mps(sites, state; linkdims=10)

U = 0.0 
V = 0.0
J = 1.0
H = H_EHM(L, J, U, V, sites)

# Start DMRG calculation:
energy, psi = dmrg(H, psi0; nsweeps, maxdim, cutoff)

S = average_single_site_entanglement_entropy(psi, L)

E_p = S # - log(L)

println("E_p = ", E_p)

After sweep 1 energy=-11.978307082021844  maxlinkdim=50 maxerr=7.06E-06 time=12.814
After sweep 2 energy=-12.05326281777596  maxlinkdim=100 maxerr=8.42E-08 time=0.317
After sweep 3 energy=-12.05334833593253  maxlinkdim=200 maxerr=1.97E-10 time=0.394
After sweep 4 energy=-12.053348366661895  maxlinkdim=400 maxerr=2.37E-14 time=0.428
After sweep 5 energy=-12.053348366664245  maxlinkdim=413 maxerr=9.89E-15 time=0.455
After sweep 6 energy=-12.05334836666427  maxlinkdim=415 maxerr=9.36E-15 time=0.439
E_p = 1.3862943611198908


In [154]:
println("dimension d = ", 4^L)
println("ln(d) = ", log(4^L))

dimension d = 1048576
ln(d) = 13.862943611198906
